# Challenge 1 — Real-Time Weather Alerts Agent

Gemini, Google Maps Geocoding, and National Weather Service. Keys are requested only at runtime and are not saved.

In [ ]:
%pip install -q --upgrade google-adk requests

import logging
import os
from getpass import getpass
from typing import Dict, List, Optional

import requests
import vertexai
from google.adk.agents import LlmAgent

PROJECT_ID = "qwiklabs-gcp-02-9e12deb8c42f"
LOCATION = "us-central1"
MODEL_GEMINI = "gemini-2.5-flash"
vertexai.init(project=PROJECT_ID, location=LOCATION)
logging.basicConfig(level=logging.INFO)
print(f"Vertex AI initialized for {PROJECT_ID} in {LOCATION}")

In [ ]:
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")
if not GOOGLE_MAPS_API_KEY:
    GOOGLE_MAPS_API_KEY = getpass("Paste your Google Maps API key: ")
    os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY

NWS_HEADERS = {"User-Agent": "ReadyNowWeatherAgent/1.0 (student lab)"}

def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Return U.S. coordinates and a formatted address from Google Maps Geocoding."""
    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json",
        params={"address": location, "components": "country:US", "key": GOOGLE_MAPS_API_KEY},
        timeout=20,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("status") != "OK" or not payload.get("results"):
        return None
    result = payload["results"][0]
    coordinates = result["geometry"]["location"]
    return {"latitude": float(coordinates["lat"]), "longitude": float(coordinates["lng"]), "formatted_address": result["formatted_address"]}

def get_extended_weather_forecast(lat: float, lon: float) -> List[Dict[str, str]]:
    """Return up to six National Weather Service forecast periods."""
    point_response = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS, timeout=20)
    point_response.raise_for_status()
    forecast_url = point_response.json()["properties"]["forecast"]
    forecast_response = requests.get(forecast_url, headers=NWS_HEADERS, timeout=20)
    forecast_response.raise_for_status()
    periods = forecast_response.json()["properties"]["periods"]
    return [{"period": p["name"], "temperature": f"{p['temperature']} {p['temperatureUnit']}", "wind": f"{p['windSpeed']} {p['windDirection']}", "forecast": p["shortForecast"], "detail": p["detailedForecast"]} for p in periods[:6]]

WEATHER_AGENT_INSTRUCTIONS = """You are Pat, a careful U.S. real-time weather-alert agent. Use get_lat_lon for a location, then get_extended_weather_forecast for its coordinates. Summarize the near-term forecast clearly. Highlight hazards such as severe storms, extreme heat, heavy snow, high winds, flooding, or fire weather. Do not invent weather information. Explain that NWS only covers U.S. locations and ask for a U.S. city/state if needed."""

weather_agent_gemini = LlmAgent(
    name="pat_weather_gemini", model=MODEL_GEMINI,
    description="Provides current National Weather Service forecasts and weather alerts for U.S. locations.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)
print("Created Gemini weather-agent definition.")

In [ ]:
# Tool tests required for Challenge 1: three U.S. locations.
TEST_LOCATIONS = ["New York, NY", "Miami, FL", "Denver, CO"]

def test_weather_tools(location: str) -> None:
    coordinates = get_lat_lon(location)
    assert coordinates is not None, f"No coordinates returned for {location}"
    forecast = get_extended_weather_forecast(coordinates["latitude"], coordinates["longitude"])
    assert forecast, f"No forecast returned for {location}"
    print(f"\n{coordinates['formatted_address']}")
    print(forecast[0])

for test_location in TEST_LOCATIONS:
    test_weather_tools(test_location)

In [ ]:
# Gemini agent-level smoke test.
from vertexai.preview import reasoning_engines

app = reasoning_engines.AdkApp(agent=weather_agent_gemini)
session = app.create_session(user_id="challenge-one-tester")
final_text = None
for event in app.stream_query(
    user_id="challenge-one-tester",
    session_id=session["id"],
    message="Give me a concise weather alert for Miami, Florida.",
):
    content = event.get("content", {})
    for part in content.get("parts", []):
        if isinstance(part, dict) and part.get("text"):
            final_text = part["text"]

if final_text:
    print(final_text)
else:
    print("Agent completed without a text response. Inspect the streamed events if needed.")